# 🏠 การพัฒนาอสังหาริมทรัพย์กระจุกตัวอยู่ตามแนวรถไฟฟ้า?

วิเคราะห์ความสัมพันธ์ระหว่างการพัฒนาอสังหาริมทรัพย์กับแนวรถไฟฟ้าในเขตกรุงเทพฯ และปริมณฑล

**คำถามที่ต้องการหาคำตอบ:**
1. การพัฒนาอสังหาริมทรัพย์กระจุกตัวตามแนวรถไฟฟ้าหรือไม่?
2. ราคาอสังหาริมทรัพย์แพงขึ้นตามระยะทางที่ใกล้รถไฟฟ้าหรือไม่?

## 1. Import Libraries

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')

## 2. Load Datasets

- **Residential Project Data**: 23,744 rows, 46 columns
- **Unit Type Data**: 43,133 rows, 28 columns
- **Train Station Data**: 318 rows, 20 columns

In [ ]:
df_project = pd.read_csv('Dataset/opendata_project.csv')
df_unittype = pd.read_csv('Dataset/opendata_unittype.csv')
df_train = pd.read_csv('Dataset/opendata_train_station.csv')

print(f'Project data: {df_project.shape}')
print(f'Unit type data: {df_unittype.shape}')
print(f'Train station data: {df_train.shape}')

## 3. Data Filtering & Preparation

กรองข้อมูลเฉพาะ:
- จังหวัดในเขตกรุงเทพฯ และปริมณฑล (กรุงเทพฯ, นนทบุรี, ปทุมธานี, สมุทรปราการ)
- เฉพาะ project_id ที่ขึ้นต้นด้วย 'project_' (ไม่มี noise)
- เฉพาะ property type: บ้านเดี่ยว, คอนโด, ทาวน์โฮม, อาคารพาณิชย์, บ้านแฝด

In [ ]:
PROVINCES_BKK = [3781, 3372, 3599, 3498]  # Bangkok, Nonthaburi, Pathum Thani, Samut Prakan
PROPERTY_TYPES = [1, 2, 3, 4, 6, 20000]

mask_province = df_project['province_id'].isin(PROVINCES_BKK)
mask_project = df_project['project_id'].str.contains('project', na=False)

cols_project = ['project_id', 'name_en', 'name_th', 'propertytype_name_en',
                'propertytype_name_th', 'price_min', 'latitude', 'longitude',
                'province_id', 'province_name_en', 'province_name_th',
                'developer_name_en', 'developer_name_th']

n_df_project = df_project.loc[mask_province & mask_project, cols_project]
n_df_unittype = df_unittype[df_unittype['propertytype_id'].isin(PROPERTY_TYPES)]
n_df_unittype = n_df_unittype[['unittype_id', 'project_id', 'propertytype_id',
                                'propertytype_name_en', 'propertytype_name_th',
                                'area_usable_min', 'price_min']]

df_merged = pd.merge(n_df_project, n_df_unittype, how='inner', on='project_id')

print(f'Filtered projects: {len(n_df_project)}')
print(f'Filtered unit types: {len(n_df_unittype)}')
print(f'Merged data: {len(df_merged)}')

## 4. วิเคราะห์ข้อที่ 1: การกระจายตัวตามแนวรถไฟฟ้า

Plot ตำแหน่งโครงการที่อยู่อาศัยเทียบกับสถานีรถไฟฟ้าบนแผนที่

In [ ]:
fig = px.scatter_mapbox(
    df_train, lat='latitude', lon='longitude',
    color='station_th', hover_name='station_th',
    hover_data=['line_type', 'line_th'],
    color_discrete_sequence=['blue'], zoom=12, height=500
)

fig2 = px.scatter_mapbox(
    n_df_project, lat='latitude', lon='longitude',
    hover_name='name_th', hover_data=['propertytype_name_en', 'price_min'],
    color_discrete_sequence=['magenta'], zoom=12, height=500
)

fig.add_trace(fig2.data[0])
fig.update_layout(mapbox_style='open-street-map')
fig.update_layout(margin={'r': 0, 't': 0, 'l': 0, 'b': 0})
fig.show()

## 5. วิเคราะห์ข้อที่ 2: ราคาอสังหาริมทรัพย์กับพื้นที่และประเภท

### 5.1 ราคาเฉลี่ยตามประเภท (Group by Property Type)

In [ ]:
t1 = df_merged.groupby('propertytype_name_th_x')['price_min_y'].mean().sort_values()
t1_name = list(t1.keys())
t1_value = t1.values / 1000000

plt.figure(figsize=(12, 5))
bars = plt.barh(range(len(t1_name)), t1_value, color='#2E86AB', edgecolor='white', height=0.6)
plt.yticks(range(len(t1_name)), t1_name, fontsize=11)
plt.xlabel('ราคาเฉลี่ย (ล้านบาท)', fontsize=12)
plt.title('ราคาเฉลี่ยตามแบบบ้าน', fontsize=14, fontweight='bold')
for i, v in enumerate(t1_value):
    plt.text(v + 0.05, i, f'{v:.2f}M', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### 5.2 ราคาเฉลี่ยตามจังหวัด (Group by Province)

In [ ]:
t2 = df_merged.groupby('province_name_th')['price_min_y'].mean().sort_values()
t2_name = list(t2.keys())
t2_value = t2.values / 1000000

plt.figure(figsize=(12, 5))
colors = ['#A63A50', '#F0A202', '#2E86AB', '#7E5F5A']
bars = plt.barh(range(len(t2_name)), t2_value, color=colors, edgecolor='white', height=0.5)
plt.yticks(range(len(t2_name)), t2_name, fontsize=12)
plt.xlabel('ราคาเฉลี่ย (ล้านบาท)', fontsize=12)
plt.title('ราคาเฉลี่ยตามพื้นที่', fontsize=14, fontweight='bold')
for i, v in enumerate(t2_value):
    plt.text(v + 0.05, i, f'{v:.2f}M', va='center', fontsize=11)
plt.tight_layout()
plt.show()

## 6. การวิเคราะห์เพิ่มเติม

### 6.1 จำนวนโครงการแยกตามประเภทในแต่ละจังหวัด

In [ ]:
ct = pd.crosstab(df_merged['province_name_th'], df_merged['propertytype_name_th_x'])
ax = ct.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2', edgecolor='white')
plt.xlabel('จังหวัด', fontsize=12)
plt.ylabel('จำนวนโครงการ', fontsize=12)
plt.title('จำนวนโครงการจำแนกตามประเภทอสังหาฯ ในแต่ละจังหวัด', fontsize=13, fontweight='bold')
plt.legend(title='ประเภท', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 6.2 ผู้พัฒนาอสังหาฯ ที่มีโครงการมากที่สุด 15 อันดับ

In [ ]:
dev_counts = n_df_project['developer_name_en'].dropna().value_counts().head(15)
plt.figure(figsize=(10, 6))
plt.barh(range(len(dev_counts)), dev_counts.values, color='#F0A202', edgecolor='white')
plt.yticks(range(len(dev_counts)), dev_counts.index, fontsize=9)
plt.xlabel('จำนวนโครงการ', fontsize=12)
plt.title('ผู้พัฒนาโครงการมากที่สุด 15 อันดับ', fontsize=13, fontweight='bold')
for i, v in enumerate(dev_counts.values):
    plt.text(v + 0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 6.3 สิ่งอำนวยความสะดวกในโครงการ

In [ ]:
facilities = ['facility_clubhouse', 'facility_fitness', 'facility_meeting',
              'facility_park', 'facility_playground', 'facility_pool', 'facility_security']
facility_names = ['คลับเฮาส์', 'ฟิตเนส', 'ห้องประชุม', 'สวนสาธารณะ',
                  'สนามเด็กเล่น', 'สระว่ายน้ำ', 'ระบบรักษาความปลอดภัย']
pcts = [df_project[col].dropna().mean() * 100 for col in facilities]

plt.figure(figsize=(10, 6))
plt.barh(range(len(facility_names)), pcts, color='#2E86AB', edgecolor='white')
plt.yticks(range(len(facility_names)), facility_names, fontsize=11)
plt.xlabel('เปอร์เซ็นต์ของโครงการที่มี (%)', fontsize=12)
plt.title('สัดส่วนโครงการที่มีสิ่งอำนวยความสะดวก', fontsize=13, fontweight='bold')
for i, v in enumerate(pcts):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### 6.4 จำนวนสถานีรถไฟฟ้าแยกตามสาย

In [ ]:
line_counts = df_train['line_th'].value_counts()
plt.figure(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(line_counts)))
plt.barh(range(len(line_counts)), line_counts.values, color=colors, edgecolor='white')
plt.yticks(range(len(line_counts)), line_counts.index, fontsize=10)
plt.xlabel('จำนวนสถานี', fontsize=12)
plt.title('จำนวนสถานีรถไฟฟ้าแยกตามสาย', fontsize=13, fontweight='bold')
for i, v in enumerate(line_counts.values):
    plt.text(v + 0.3, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 7. สรุปผลการวิเคราะห์

### คำถามที่ 1: อสังหาฯ กระจุกตัวตามแนวรถไฟฟ้าหรือไม่?
- การกระจายตัวของอสังหาฯ **ไม่ได้กระจุกตัวตามแนวรถไฟฟ้าทั้งหมด** แต่ขึ้นอยู่กับประเภท
- **คอนโดมิเนียม** มีแนวโน้มกระจุกตัวตามแนวรถไฟฟ้ามากที่สุด
- **บ้านเดี่ยว** มักตั้งอยู่ในพื้นที่ห่างจากรถไฟฟ้า เนื่องจากต้องการพื้นที่ขนาดใหญ่

### คำถามที่ 2: ราคาแพงขึ้นตามระยะทางใกล้รถไฟฟ้าหรือไม่?
- กรุงเทพฯ มีราคาเฉลี่ยสูงสุด รองลงมาคือสมุทรปราการและนนทบุรี
- บ้านเดี่ยวมีราคาเฉลี่ยสูงที่สุดในทุกจังหวัด
- คอนโดในกรุงเทพฯ มีราคาสูงกว่าในจังหวัดอื่นอย่างชัดเจน

### แนวทางการวิเคราะห์เพิ่มเติม
- คำนวณระยะทาง Haversine จากโครงการไปยังสถานีรถไฟฟ้าที่ใกล้ที่สุด
- วิเคราะห์ correlation ระหว่างราคากับ distance
- ใช้ Heatmap แสดง clustering แทน scatter plot
- สร้าง ML Model เพื่อทำนายราคาจาก distance, property type, facilities